In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 72.0 MB/s eta 0:00:00


In [19]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [4]:
import gensim.downloader as api
nlp=api.load("glove-wiki-gigaword-100")

[==================================================] 100.0% 128.1/128.1MB downloaded


In [10]:
data=pd.read_csv("Fake_Real_Data.csv")
data.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


In [12]:
data['label_num']=data.label.map({'Fake':0,'Real':1})
print(data.head())

                                                Text label  label_num
0   Top Trump Surrogate BRUTALLY Stabs Him In The...  Fake          0
1  U.S. conservative leader optimistic of common ...  Real          1
2  Trump proposes U.S. tax overhaul, stirs concer...  Real          1
3   Court Forces Ohio To Allow Millions Of Illega...  Fake          0
4  Democrats say Trump agrees to work on immigrat...  Real          1


In [14]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 130.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [15]:
import spacy
model=spacy.load('en_core_web_sm')

def preprocess(text):
  doc=model(text)
  filtered_tokens=[]
  for token in doc:
    if token.is_stop or token.is_punct:
      continue
    filtered_tokens.append(token.lemma_)
  return nlp.get_mean_vector(filtered_tokens)


In [16]:
data['vector']=data['Text'].apply(preprocess)
data.head()

,Text,label,label_num,vector
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0,"[-0.015012949, 0.049764723, 0.07989554, -0.039..."
1,U.S. conservative leader optimistic of common ...,Real,1,"[-0.034071855, 0.045250613, 0.028832512, -0.00..."
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1,"[-0.009542668, 0.04375058, 0.039975125, -0.019..."
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,0,"[-0.01771663, 0.015048098, 0.054996982, -0.046..."
4,Democrats say Trump agrees to work on immigrat...,Real,1,"[-0.008493272, 0.019276159, 0.029159553, -0.01..."


In [22]:
X=np.stack(data.vector)

In [23]:
X_train,X_test,y_train,y_test=train_test_split(X,data.label_num,test_size=.2,random_state=42,stratify=data.label_num)

In [24]:
model_lr=LogisticRegression()
model_lr.fit(X_train,y_train)
y_pred=model_lr.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1000
           1       0.94      0.94      0.94       980

    accuracy                           0.94      1980
   macro avg       0.94      0.94      0.94      1980
weighted avg       0.94      0.94      0.94      1980

